# Create annotation file (unnecessary)

In [ ]:
import pandas as pd

import json

In [ ]:
df = pd.read_csv("../data/input/minimal_pairs_sample_ref.tsv", sep="\t")

In [ ]:
df = df.rename(columns={"NK": "nk", "SK": "sk"})

In [ ]:
# add id
df.insert(0, "id", range(1, len(df) + 1))

In [ ]:
# manually define dialect-specific span mappings

span_annotations = {

    1: [
        {"nk": "창가림", "sk": "커튼"},
        {"nk": "들여다볼수", "sk": "들여다볼 수"},
        {"nk": "있을만큼", "sk": "있을 만큼"},
        {"nk": "남겨놓았소", "sk": "남겨 놓았소"}
    ],

    2: [
        {"nk": "초불", "sk": "촛불"},
        {"nk": "켜둔다는것", "sk": "켜둔다는 것"}
    ],

    3: [{"nk": "웨쳤습니다", "sk": "외쳤습니다"}],

    4: [
        {"nk": "일찌기", "sk": "일찍"},
        {"nk": "생각할만큼", "sk": "생각할 만큼"},
        {"nk": "녀자", "sk": "여자"},
        {"nk": "여러번", "sk": "여러 번"},
        {"nk": "알고보면", "sk": "알고 보면"},
        {"nk": "보내고있었다는것", "sk": "보내고 있었다는 것"}
    ],

    5: [
        {"nk": "요우", "sk": "요 위"},
        {"nk": "하불밖", "sk": "이불 밖"}
    ],

    6: [
        {"nk": "우습강스런데", "sk": "우스꽝스러운 점"},
        {"nk": "삼고있었지만", "sk": "삼고 있었지만"},
        {"nk": "안가서", "sk": "안 가서"},
        {"nk": "고쳐주는데", "sk": "고쳐 주는 데"},
        {"nk": "되였다", "sk": "되었다"}
    ],

    7: [{"nk": "지팽이", "sk": "지팡이"}],

    8: [{"nk": "수양", "sk": "숫양"}],

    9: [{"nk": "수집어하였을가", "sk": "수줍어하였을까"}],

    10: [{"nk": "안해", "sk": "아내"}],

    11: [{"nk": "래일", "sk": "내일"}],

    12: [
        {"nk": "뛰여", "sk": "뛰어"},
        {"nk": "책상밑", "sk": "책상 밑"},
        {"nk": "기여", "sk": "기어"},
        {"nk": "난로옆", "sk": "난로 옆"}
    ],

    13: [
        {"nk": "되여", "sk": "되어"},
        {"nk": "밝았을때", "sk": "밝았을 때"},
        {"nk": "발견되였다", "sk": "발견되었다"},
    ],

    14: [
        {"nk": "녀자", "sk": "여자"},
        {"nk": "원탁곁", "sk": "원탁 곁"},
        {"nk": "사진첩우", "sk": "사진첩 위"},
        {"nk": "들여다보고있었는데", "sk": "들여다보고 있었는데"},
        {"nk": "기다리는듯싶었다", "sk": "기다리는 듯 싶었다"}
    ],

    15: [
        {"nk": "땅우", "sk": "땅 위"},
        {"nk": "사라지고말리라", "sk": "사라지고 말리라"}
    ],

    16: [{"nk": "리해할수", "sk": "이해할 수"}],

    17: [
        {"nk": "이딸리아", "sk": "이탈리아"},
        {"nk": "아름다와요", "sk": "아름다워요"},
    ],

    18: [
        {"nk": "사는것", "sk": "사는 것"},
        {"nk": "기뻐뛰여라", "sk": "기뻐 뛰어라"}
    ],

    19: [
        {"nk": "수자", "sk": "숫자"},
        {"nk": "표시하는것", "sk": "표시하는 것"},
    ],

    20: [{"nk": "인차", "sk": "이내"}]

}

In [ ]:
df["spans_json"] = df["id"].map(lambda i: json.dumps(span_annotations[i], ensure_ascii=False))

In [ ]:
df.to_csv("../data/output/minimal_pairs_sample_annotations.tsv", sep="\t", index=False)

# Dialect-aware chrF Score

Instead of treating each sentence as a single evaluation unit, treat annotated dialect-specific spans (and, separately, dialect-invariant spans) as the evaluation units while keeping the standard corpus-level chrF computation unchanged. This is possible because corpus-level chrF operates on a collection of text segments rather than requiring complete sentences.
1.	Separate sentences into dialect-specific and dialect-invariant parts
2.	Compute chrF score only on the parts of the string that are NK or SK specific 
= dialect-specific score
3.	Compute chrF score on parts of the string that should be identical (not NK or SK specific)
= dialect-invariant score
4.	Compute final chrF score that takes into account both the dialect-specific score & dialect-invariant score


In [4]:
import json
import re
import pandas as pd
from difflib import SequenceMatcher
from sacrebleu.metrics import CHRF

In [2]:
chrf = CHRF(word_order=0, beta=3)

In [3]:
def chrf3(hyp: str, ref: str) -> float:
    if not hyp.strip() or not ref.strip(): # empty strings return 0
        return 0.0
    return chrf.sentence_score(hyp, [ref]).score

In [6]:
def remove_span_by_offsets(text: str, start: int, end: int) -> str:

    """
    Remove one span from text using its character offsets.
    The text before and after the removed span is joined to form the remaining residual text.
    """
    
    return (text[:start] + text[end:]).strip()

Find the single longest substring that appears identically in both the reference and hypothesis, call that the dialect-invariant part, and treat everything left over as the dialect-specific residual.

Example:

Reference: 이탈리아 거긴 정말 경치가 아름다워요.

Hypothesis: 이딸리아는 정말 경치가 아름다워요.

The longest identical contiguous substring: 정말 경치가 아름다워요.


split_invariant_first() separates the sentence pair into:

Invariant: 정말 경치가 아름다워요.

Reference residual: 이탈리아 거긴

Hypothesis residual: 이딸리아는

In [ ]:
def split_invariant_first(ref: str, hyp: str):

    """
    Identify the longest identical contiguous block between the reference and hypothesis as the dialect-invariant part.
    Everything outside that shared block is treated as the dialect-specific part.
    """

    matcher = SequenceMatcher(None, ref, hyp, autojunk=False)

    # find the longest contiguous block that is identical in both strings
    match = matcher.find_longest_match(0, len(ref), 0, len(hyp))

    # match.a: where the shared block begins in the reference
    # match.b: where the shared block begins in the hypothesis
    # match.size: how many characters the shared block contains
    
    # if no common characters were found
    if match.size == 0:
        return {"ref_invariant": "", "hyp_invariant": "", 
                "ref_dialect_specific": ref.strip(), "hyp_dialect_specific": hyp.strip(),}

    # convert the match into character offsets to define the matching slices
    ref_start = match.a
    ref_end = match.a + match.size
    hyp_start = match.b
    hyp_end = match.b + match.size

    # extract the invariant text
    ref_invariant = ref[ref_start:ref_end].strip()
    hyp_invariant = hyp[hyp_start:hyp_end].strip()
    # the reference and hypothesis invariant strings should contain the same text
    
    # remove the invariant block from each sentence
    ref_dialect_specific = remove_span_by_offsets(ref, ref_start, ref_end) # 이탈리아 거긴 [정말 경치가 아름다워요.]
    hyp_dialect_specific = remove_span_by_offsets(hyp, hyp_start, hyp_end) # 이딸리아는 [정말 경치가 아름다워요.]
    # the remaining text is treated as dialect-specific
    
    return {
        "ref_invariant": ref_invariant,
        "hyp_invariant": hyp_invariant,
        "ref_dialect_specific": ref_dialect_specific,
        "hyp_dialect_specific": hyp_dialect_specific,
    }

Evaluate each sentence pair

In [8]:
def evaluate_one_example(ref: str, hyp: str):
    """
    Compute:
    1. whole-sentence chrF3
    2. dialect-specific chrF3
    3. dialect-invariant chrF3
    4. combined dialect-aware chrF3
    """
    parts = split_invariant_first(ref, hyp)

    ref_invariant = parts["ref_invariant"]
    hyp_invariant = parts["hyp_invariant"]
    
    ref_dialect_specific = parts["ref_dialect_specific"]
    hyp_dialect_specific = parts["hyp_dialect_specific"]
    
    # dialect-invariant chrF3 : evaluates whether the model translated the non-dialect-specific parts correctly
    dialect_invariant_score = chrf3(hyp_invariant, ref_invariant)

    # dialect_specific chrF3 : evaluates whether the model translated the dialect-specific parts correctly
    dialect_specific_score = chrf3(hyp_dialect_specific, ref_dialect_specific)

    # compute whole-sentence chrF3
    whole_sentence_score = chrf3(hyp, ref)

    # combined chrF3 score: tune alpha depending on how much I want to prioritize dialect conversion
    alpha = 0.7
    combined_score = (
        alpha * dialect_specific_score
        + (1 - alpha) * dialect_invariant_score
    )

    return {
        "whole_chrF3": whole_sentence_score,
        "dialect_specific_chrF3": dialect_specific_score,
        "dialect_invariant_chrF3": dialect_invariant_score,
        "combined_dialect_aware_chrF3": combined_score,
        "ref_dialect_spans": ref_dialect_specific,
        "hyp_dialect_spans": hyp_dialect_specific,
        "ref_invariant": ref_invariant,
        "hyp_invariant": hyp_invariant,
    }


In [9]:
def evaluate_outputs(ref_path, hyp_path, save_path):
    ref = pd.read_csv(ref_path, sep="\t")
    hyp = pd.read_csv(hyp_path, sep="\t")

    # merge the input sentence pairs (gold standard with annotations) with MT outputs
    df = ref.merge(hyp, on="id", how="left")

    rows = []

    for _, row in df.iterrows():
        result = evaluate_one_example(
            ref=row["sk"],
            hyp=row["hyp"],
            # spans_json=row["spans_json"]
        )

        rows.append({
            "id": row["id"],
            "nk": row["nk"],
            "ref_sk": row["sk"],
            "hyp_sk": row["hyp"],
            **result
        })

    result_df = pd.DataFrame(rows)
    result_df.to_csv(save_path, sep="\t", index=False)

    # average sentence-level scores across all examples
    summary = result_df[
        [
            "whole_chrF3",
            "dialect_specific_chrF3",
            "dialect_invariant_chrF3",
            "combined_dialect_aware_chrF3"
        ]
    ].mean()

    print("\n===== Mean Sentence-level Results =====")
    print(summary)

    return result_df, summary

In [10]:
if __name__ == "__main__":
    evaluate_outputs(
        ref_path="../data/output/minimal_pairs_sample_annotations.tsv",
        hyp_path="../data/input/minimal_pairs_sample_hyp.tsv",
        save_path="../data/output/dialect_aware_chrf_results_solution2.tsv"
    )


===== Mean Sentence-level Results =====
whole_chrF3                      79.584544
dialect_specific_chrF3           58.921572
dialect_invariant_chrF3         100.000000
combined_dialect_aware_chrF3     71.245100
dtype: float64
